# Mock Serving API Walkthrough

This notebook is for reading `mock_serving_api.py` as a data-serving API.

The teaching design has two layers. The normal business-style endpoints, such as `/api/v1/sources/taxi/records`, show what a clean provider contract looks like. The `/debug-lab/...` challenge endpoints deliberately expose one problem at a time: auth failure, schema change, rate limit, flaky server, duplicates, pagination, timeout, checkpointing, incremental loading, or scheduled ingestion. Students should call a challenge URL, inspect the status code or response body, and repair the client. Admin controls such as `/admin/scenario/...` exist mainly for instructor debugging and provider-side demonstrations.

It connects to the Day 2 PPT section:

- API as Data Serving
- FastAPI basics
- API contract design
- Schema and validation
- Error handling
- Provider-side protection

The goal is not to memorize the code. The goal is to see how a data engineer can design an API that serves real data while protecting both the provider and the consumer.


## Setup

Start the API in a terminal:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

Then keep that terminal open.

Open the interactive API documentation:

```text
http://127.0.0.1:8011/docs
```


In [23]:
from pathlib import Path
import inspect
import json

import requests

BASE_URL = "http://127.0.0.1:8011"
SOURCE_FILE = Path("mock_serving_api.py")

print("Mock Serving API source exists:", SOURCE_FILE.exists())


Mock Serving API source exists: True


## 1. API as Data Serving

In Day 1, we consumed real APIs directly.

In Day 2, `mock_serving_api.py` becomes a provider-side API layer:

```text
access API notebook
      ->
mock_serving_api.py
      ->
real data.gov.sg APIs
```

This is a common pattern in production. A company may not expose raw database tables or raw upstream APIs directly. Instead, it builds a serving API with a stable contract.


In [24]:
response = requests.get(f"{BASE_URL}/api/v1/status", timeout=10)
response.raise_for_status()
status = response.json()
print(json.dumps(status, indent=2)[:1500])


{
  "scenario": "flaky",
  "schema_version": "v1",
  "request_count": 921,
  "last_request_at": "2026-06-20T06:47:49+00:00",
  "available_scenarios": [
    "normal",
    "flaky",
    "rate_limit",
    "timeout",
    "http_error",
    "duplicates"
  ],
  "available_schema_versions": [
    "v1",
    "v2"
  ],
  "upstream_sources": {
    "taxi": "https://api.data.gov.sg/v1/transport/taxi-availability",
    "weather": "https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast",
    "rainfall": "https://api-open.data.gov.sg/v2/real-time/api/rainfall"
  },
  "upstream_cache": {
    "taxi": {
      "fetched_at": "2026-06-20T06:47:01+00:00",
      "age_seconds": 204.0
    },
    "weather": {
      "fetched_at": "2026-06-20T06:29:14+00:00",
      "age_seconds": 1271.9
    },
    "rainfall": {
      "fetched_at": "2026-06-20T06:46:31+00:00",
      "age_seconds": 235.1
    }
  },
  "upstream_cache_ttl_seconds": 60,
  "rate_limit": {
    "max_requests": 3,
    "window_seconds": 5
  }
}


## 2. FastAPI Basics

FastAPI turns Python functions into HTTP endpoints.

### What Is an Endpoint?

An **endpoint** is one specific access point of an API.

It is usually defined by four things:

| Part | Example |
|---|---|
| HTTP method | `GET` |
| URL path | `/api/v1/status` |
| Handler function | `status()` |
| Request/response contract | accepted parameters and returned JSON fields |

So this is one endpoint:

```text
GET /api/v1/status
```

And this is another endpoint:

```text
GET /api/v1/sources/taxi/records?page=1&page_size=10
```

The path tells the user **what resource or action** they are requesting. The Python function tells FastAPI **what code should run** when that request arrives.

In the source code, look for:

```python
app = FastAPI(...)

@app.get("/api/v1/status")
def status():
    ...
```

Key mapping:

| Code | Meaning |
|---|---|
| `FastAPI()` | Create the API application |
| `@app.get("/api/v1/status")` | Register a `GET /api/v1/status` endpoint |
| `status()` | Handler function for that endpoint |
| Function parameters | Query/header/path inputs |
| Return dictionary | JSON response |

A good endpoint should answer one clear question. For example:

```text
GET /api/v1/status
```

answers: "Is this API running and what mode is it in?"

```text
GET /api/v1/sources/taxi/records
```

answers: "Give me flat taxi records, with pagination."


In [25]:
import mock_serving_api

print(mock_serving_api.app.title)
print(mock_serving_api.app.version)

routes = []
for route in mock_serving_api.app.routes:
    if hasattr(route, "methods") and hasattr(route, "path"):
        routes.append((sorted(route.methods), route.path))

for methods, route_path in routes[:20]:
    print(methods, route_path)


Day 2.1 Mock Serving API
1.0.0
['GET', 'HEAD'] /openapi.json
['GET', 'HEAD'] /docs
['GET', 'HEAD'] /docs/oauth2-redirect
['GET', 'HEAD'] /redoc
['GET'] /
['GET'] /api/v1/status
['POST'] /admin/scenario/{scenario}
['POST'] /admin/schema/{schema_version}
['POST'] /admin/reset
['GET'] /api/v1/transport/taxi-availability
['GET'] /api/v1/environment/2-hour-weather-forecast
['GET'] /api/v1/environment/rainfall
['GET'] /api/v1/sources/{source}/records
['GET'] /api/v1/taxi/availability
['GET'] /debug-lab
['GET'] /debug-lab/01-auth-moved/taxi/records
['GET'] /debug-lab/02-schema-change/taxi/records
['GET'] /debug-lab/03-rate-limit/weather/records
['GET'] /debug-lab/04-flaky/rainfall/records
['GET'] /debug-lab/05-duplicates/taxi/records


## 3. Raw Proxy Endpoints

The API keeps Day 1-compatible raw endpoints:

```text
/api/v1/transport/taxi-availability
/api/v1/environment/2-hour-weather-forecast
/api/v1/environment/rainfall
```

These endpoints preserve the upstream response shape. That means Day 1 parsing code can still work.

Design idea:

A serving API can expose a compatibility layer for old clients while also offering new cleaner endpoints.


In [26]:
for endpoint in [
    "/api/v1/transport/taxi-availability",
    "/api/v1/environment/2-hour-weather-forecast",
    "/api/v1/environment/rainfall",
]:
    response = requests.get(f"{BASE_URL}{endpoint}", timeout=15)
    print(endpoint, response.status_code, list(response.json().keys())[:5])


/api/v1/transport/taxi-availability 200 ['type', 'crs', 'features']
/api/v1/environment/2-hour-weather-forecast 200 ['code', 'data', 'errorMsg']
/api/v1/environment/rainfall 200 ['code', 'data', 'errorMsg']


## 4. Flat Data-Serving Endpoint

The main data-serving endpoint is:

```text
GET /api/v1/sources/{source}/records
```

Examples:

```text
/api/v1/sources/taxi/records?page=1&page_size=10
/api/v1/sources/weather/records?page=1&page_size=10
/api/v1/sources/rainfall/records?page=1&page_size=10
```

This endpoint converts nested upstream JSON into flat rows that are easier for analytics ingestion.


In [27]:
# Keep this walkthrough section independent from earlier challenge/admin state.
requests.post(f"{BASE_URL}/admin/scenario/normal", timeout=10).raise_for_status()
requests.post(f"{BASE_URL}/admin/schema/v1", timeout=10).raise_for_status()


In [28]:
for source in ["taxi", "weather", "rainfall"]:
    response = requests.get(
        f"{BASE_URL}/api/v1/sources/{source}/records",
        params={"page": 1, "page_size": 3, "client_id": "walkthrough"},
        timeout=15,
    )
    response.raise_for_status()
    payload = response.json()
    print("SOURCE:", source)
    print("total_records:", payload["total_records"])
    print("first row:")
    print(json.dumps(payload["data"][0], indent=2)[:800])


SOURCE: taxi
total_records: 1964
first row:
{
  "record_id": "taxi:2026-06-20T14:49:58+08:00:1",
  "api_timestamp": "2026-06-20T14:49:58+08:00",
  "taxi_index": 1,
  "longitude": 103.6476,
  "latitude": 1.32231,
  "available_taxi_count": 1964,
  "source": "data.gov.sg taxi availability"
}
SOURCE: weather
total_records: 47
first row:
{
  "record_id": "weather:2026-06-20T14:36:42+08:00:Ang Mo Kio",
  "api_timestamp": "2026-06-20T14:30:00+08:00",
  "api_update_timestamp": "2026-06-20T14:36:42+08:00",
  "area": "Ang Mo Kio",
  "forecast": "Partly Cloudy (Day)",
  "valid_period_start": "2026-06-20T14:30:00+08:00",
  "valid_period_end": "2026-06-20T16:30:00+08:00",
  "source": "data.gov.sg two-hour weather forecast"
}
SOURCE: rainfall
total_records: 77
first row:
{
  "record_id": "rainfall:2026-06-20T14:40:00+08:00:S218",
  "reading_timestamp": "2026-06-20T14:40:00+08:00",
  "station_id": "S218",
  "station_name": "Bukit Batok Street 34",
  "longitude": 103.75065,
  "latitude": 1.36491,
  "r

## 5. API Contract and Pagination

A serving API should not return unlimited data.

This response includes metadata:

```json
{
  "source": "taxi",
  "page": 1,
  "page_size": 10,
  "total_records": 3000,
  "total_pages": 300,
  "next_page": 2,
  "data": [...]
}
```

Teaching point:

The response contract tells the client how to continue. The client should loop until `next_page` is `None`.


In [29]:
print(inspect.getsource(mock_serving_api.paginated_response))


def paginated_response(
    records: list[dict[str, object]],
    page: int,
    page_size: int,
    source: SourceName,
    scenario: Scenario,
    schema_version: SchemaVersion,
) -> dict[str, object]:
    total_records = len(records)
    total_pages = (total_records + page_size - 1) // page_size
    start = (page - 1) * page_size
    end = start + page_size
    page_records = records[start:end]

    return {
        "source": source,
        "page": page,
        "page_size": page_size,
        "total_records": total_records,
        "total_pages": total_pages,
        "next_page": page + 1 if page < total_pages else None,
        "schema_version": schema_version,
        "scenario": scenario,
        "data": page_records,
    }



## 6. Input Validation

FastAPI can validate query parameters before the endpoint runs.

In `paginated_source_records`, observe:

```python
page: int = Query(default=1, ge=1)
page_size: int = Query(default=10, ge=1, le=20)
```

This protects the API from invalid or expensive requests such as:

```text
?page=-1
?page_size=1000000
```


In [30]:
bad_response = requests.get(
    f"{BASE_URL}/api/v1/sources/taxi/records",
    params={"page": -1, "page_size": 1000000},
    timeout=10,
)
print("status:", bad_response.status_code)
print(json.dumps(bad_response.json(), indent=2)[:1200])


status: 422
{
  "detail": [
    {
      "type": "greater_than_equal",
      "loc": [
        "query",
        "page"
      ],
      "msg": "Input should be greater than or equal to 1",
      "input": "-1",
      "ctx": {
        "ge": 1
      }
    },
    {
      "type": "less_than_equal",
      "loc": [
        "query",
        "page_size"
      ],
      "msg": "Input should be less than or equal to 20",
      "input": "1000000",
      "ctx": {
        "le": 20
      }
    }
  ]
}


## 7. Upstream Protection: Cache and Single-Flight Lock

This service uses real data.gov.sg APIs. If every student refreshes repeatedly, we do not want every request to hit the real upstream provider.

Provider-side protection in this file:

- `UPSTREAM_CACHE_TTL_SECONDS`: cache real upstream responses in memory.
- `upstream_locks`: only one request per source refreshes the cache at a time.
- stale cache fallback: if upstream fails but old cache exists, return old cache.

This is provider-side responsibility: protect the upstream provider and keep the classroom service stable.


In [31]:
print(inspect.getsource(mock_serving_api.fetch_upstream_payload))


def fetch_upstream_payload(source: SourceName) -> dict[str, object]:
    """
    Fetch a real upstream payload with cache and stampede protection.

    Classroom safety rule:
    - Client requests should usually hit our in-memory cache.
    - Only one request per source is allowed to refresh the cache at a time.
    - If the upstream API fails but we have an older cached payload, return it.

    This keeps the exercise realistic without turning the class into a traffic
    spike against data.gov.sg.
    """
    cached = upstream_cache.get(source)
    now = time.time()
    if cached and now - float(cached["fetched_at_epoch"]) < UPSTREAM_CACHE_TTL_SECONDS:
        return cached["payload"]  # type: ignore[return-value]

    with upstream_locks[source]:
        # Another request may have refreshed the cache while this request was
        # waiting for the lock, so check the cache again before calling upstream.
        cached = upstream_cache.get(source)
        now = time.time()
        if

## 8. Schema Change as a Provider-Side Contract Problem

The API can intentionally switch schema version:

```text
v1: available_taxi_count
v2: taxi_count
```

This demonstrates that field names are part of the API contract.

A real provider should avoid silent breaking changes. If a breaking change is necessary, use versioning such as `/api/v2/...` or provide a migration period.


In [32]:
response = requests.get(
    f"{BASE_URL}/debug-lab/02-schema-change/taxi/records",
    params={"page": 1, "page_size": 1},
    timeout=15,
)
response.raise_for_status()
print(json.dumps(response.json()["data"][0], indent=2))


{
  "record_id": "taxi:2026-06-20T14:49:58+08:00:1",
  "api_timestamp": "2026-06-20T14:49:58+08:00",
  "taxi_index": 1,
  "longitude": 103.6476,
  "latitude": 1.32231,
  "source": "data.gov.sg taxi availability",
  "taxi_count": 1964,
  "schema_note": "schema v2 renamed available_taxi_count to taxi_count"
}


## 9. Error Handling and Failure Simulation

The service intentionally simulates common API problems:

| Scenario | HTTP signal | Client technique |
|---|---|---|
| `flaky` | `503` | retry with backoff |
| `rate_limit` | `429` + `Retry-After` | slow down and retry later |
| `timeout` | client timeout exception | timeout handling and retry |
| `http_error` | `503` | classify temporary error |
| `duplicates` | successful but wrong data | dedup / quality check |

Important idea:

The provider can return useful status codes and messages. The consumer must read them and react correctly.


In [33]:
for endpoint in [
    "/debug-lab/01-auth-moved/taxi/records",
    "/debug-lab/03-rate-limit/weather/records",
    "/debug-lab/04-flaky/rainfall/records",
]:
    response = requests.get(
        f"{BASE_URL}{endpoint}",
        params={"page": 1, "page_size": 1, "client_id": "walkthrough_error_demo"},
        timeout=15,
    )
    print("endpoint:", endpoint)
    print("status:", response.status_code)
    print(json.dumps(response.json(), indent=2)[:800])


endpoint: /debug-lab/01-auth-moved/taxi/records
status: 401
{
  "detail": "Missing api_key query parameter. Expected api_key=class-demo-key."
}
endpoint: /debug-lab/03-rate-limit/weather/records
status: 200
{
  "source": "weather",
  "page": 1,
  "page_size": 1,
  "total_records": 47,
  "total_pages": 47,
  "next_page": 2,
  "schema_version": "v1",
  "scenario": "rate_limit",
  "data": [
    {
      "record_id": "weather:2026-06-20T14:36:42+08:00:Ang Mo Kio",
      "api_timestamp": "2026-06-20T14:30:00+08:00",
      "api_update_timestamp": "2026-06-20T14:36:42+08:00",
      "area": "Ang Mo Kio",
      "forecast": "Partly Cloudy (Day)",
      "valid_period_start": "2026-06-20T14:30:00+08:00",
      "valid_period_end": "2026-06-20T16:30:00+08:00",
      "source": "data.gov.sg two-hour weather forecast"
    }
  ]
}
endpoint: /debug-lab/04-flaky/rainfall/records
status: 503
{
  "detail": "Temporary server error. This client succeeds after two failed attempts. attempt=1"
}


## 10. `/debug-lab`: A Teaching Index Endpoint

`/debug-lab` is not a normal business data endpoint.

It is a **teaching index endpoint**. Its job is to return a small JSON menu that lists the fixed debugging challenges.

When you open:

```text
GET /debug-lab
```

the API returns something like:

```json
{
  "idea": "Each endpoint maps to one row in the PPT robust API collection table.",
  "endpoints": {
    "01_auth_moved": "/debug-lab/01-auth-moved/taxi/records",
    "02_schema_changed": "/debug-lab/02-schema-change/taxi/records"
  }
}
```

So there are two levels:

| Endpoint type | Example | Purpose |
|---|---|---|
| Index endpoint | `GET /debug-lab` | Show the list of available challenges |
| Challenge endpoint | `GET /debug-lab/03-rate-limit/weather/records` | Trigger one fixed API problem |

Why design it this way?

- Learners can discover the available exercises from one place.
- Each challenge URL has one stable behavior.
- Different groups can work on different API problems at the same time.
- The instructor does not need to keep changing global API mode.

In production, we usually would not expose a `/debug-lab` endpoint. This exists only because this API is also a teaching tool.


In [34]:
response = requests.get(f"{BASE_URL}/debug-lab", timeout=10)
response.raise_for_status()
print(json.dumps(response.json(), indent=2))


{
  "idea": "Each endpoint maps to one row in the PPT robust API collection table.",
  "expected_api_key": "class-demo-key",
  "endpoints": {
    "01_auth_moved": "/debug-lab/01-auth-moved/taxi/records",
    "02_schema_changed": "/debug-lab/02-schema-change/taxi/records",
    "03_rate_limited": "/debug-lab/03-rate-limit/weather/records",
    "04_flaky_server": "/debug-lab/04-flaky/rainfall/records",
    "05_duplicates": "/debug-lab/05-duplicates/taxi/records",
    "06_pagination": "/debug-lab/06-pagination/taxi/records",
    "07_timeout": "/debug-lab/07-timeout/taxi/records",
    "08_checkpoint": "/debug-lab/08-checkpoint/taxi/records",
    "09_incremental": "/debug-lab/09-incremental/weather/records",
    "10_scheduled_ingestion": "/debug-lab/10-scheduled/taxi/current"
  }
}


## 11. Summary: When You Provide an API

When we provide an API to other users, dashboards, or systems, we are publishing a contract.

A good data-serving API should pay attention to these points:

| Area | What to design |
|---|---|
| Clear purpose | Each endpoint should answer one clear data question. |
| Stable contract | Keep field names and response structure predictable. |
| Versioning | Use versions such as `/api/v1/...` when contracts may change. |
| Input validation | Reject invalid or expensive parameters before running heavy work. |
| Pagination | Do not return unlimited data in one response. |
| Freshness | Tell users when the data was last updated. |
| Error handling | Return useful status codes and messages such as `401`, `429`, `503`. |
| Provider protection | Use cache, rate limits, and concurrency controls to protect the service. |
| Upstream protection | Do not blindly forward every client request to an upstream API. |
| Observability | Log requests, status codes, duration, and failures. |

Key takeaway:

```text
A data API is not just a Python function that returns JSON.
It is a reliability boundary between data producers and data consumers.
```

In this mock serving API:

- Raw proxy endpoints preserve Day 1 compatibility.
- Flat `/api/v1/sources/{source}/records` endpoints provide cleaner serving contracts.
- Validation protects the provider from bad requests.
- Cache and single-flight locks protect the real upstream APIs.
- Debug endpoints intentionally simulate failures so consumers can learn robust collection.

As a data engineer, you need both perspectives:

```text
Consumer side: build clients that survive API problems.
Provider side: build APIs that are clear, stable, protected, and observable.
```
